# 🇺🇸 USA Population Growth Rate Predictor

**Dataset:** FRED (Federal Reserve Economic Data)  
**Period:** 1961 – 2022 (62 data points)  
**Goal:** Predict future US population growth rates using Machine Learning

---

### 📌 What we will do:
1. Load and explore the dataset
2. Create useful features (lag values, rolling averages)
3. Train 3 different ML models
4. Compare their performance
5. Forecast the next 10 years (2023–2032)


## Step 1 — Install & Import Libraries

> **Beginner tip:** Libraries are pre-built toolboxes. `pandas` handles data tables, `sklearn` has ML algorithms, `matplotlib` draws charts.

In [ ]:
# Install required packages (only needed once in Google Colab)
# !pip install scikit-learn pandas matplotlib numpy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

import pickle
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries loaded successfully!')

## Step 2 — Load the Dataset

> The dataset has 2 columns: **DATE** (year) and **SPPOPGROWUSA** (US population growth rate in %)

In [ ]:
# ── If running in Google Colab, upload the CSV first using Files panel (left sidebar)
# ── Or clone from GitHub:
# !git clone https://github.com/YOUR_USERNAME/car-mileage.git

df = pd.read_csv('data/Population-Growth.csv')  # change path if needed

print(f'Dataset shape: {df.shape}')
print(f'Years covered: {df["DATE"].min()[:4]} to {df["DATE"].max()[:4]}')
print()
df.head(10)

## Step 3 — Explore the Data (EDA)

> EDA = Exploratory Data Analysis. We look at the data before modelling to understand patterns.

In [ ]:
# Basic statistics
print('📊 Summary Statistics:')
print(df['SPPOPGROWUSA'].describe())

print(f'\n📉 Lowest  growth rate: {df["SPPOPGROWUSA"].min():.4f}% (in {df.loc[df["SPPOPGROWUSA"].idxmin(), "DATE"][:4]})')
print(f'📈 Highest growth rate: {df["SPPOPGROWUSA"].max():.4f}% (in {df.loc[df["SPPOPGROWUSA"].idxmax(), "DATE"][:4]})')

In [ ]:
# Plot the historical data
df['Year'] = pd.to_datetime(df['DATE']).dt.year

plt.figure(figsize=(13, 5))
plt.plot(df['Year'], df['SPPOPGROWUSA'], color='steelblue', linewidth=2.5, marker='o', markersize=4)
plt.fill_between(df['Year'], df['SPPOPGROWUSA'], alpha=0.15, color='steelblue')
plt.title('USA Population Growth Rate (1961–2022)', fontsize=14, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Growth Rate (%)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 4 — Feature Engineering

> **Beginner tip:** Feature Engineering = creating new input columns that help the model learn patterns.
> - **Lag features** = previous years' values (e.g., what was the growth 1, 2, 3 years ago?)
> - **Rolling averages** = trend over last 3 or 5 years

In [ ]:
# Lag features
df['Lag_1'] = df['SPPOPGROWUSA'].shift(1)
df['Lag_2'] = df['SPPOPGROWUSA'].shift(2)
df['Lag_3'] = df['SPPOPGROWUSA'].shift(3)

# Rolling averages
df['Rolling_3yr_avg'] = df['SPPOPGROWUSA'].rolling(window=3).mean()
df['Rolling_5yr_avg'] = df['SPPOPGROWUSA'].rolling(window=5).mean()

# Drop rows with NaN (first few rows won't have lag values)
df = df.dropna()

print(f'✅ Features created! Working dataset: {df.shape[0]} rows')
df[['Year','SPPOPGROWUSA','Lag_1','Lag_2','Lag_3','Rolling_3yr_avg','Rolling_5yr_avg']].head()

## Step 5 — Train / Test Split

> **Beginner tip:** We split data into training (model learns from this) and testing (we check how well it learned).
> - 80% = training
> - 20% = testing

In [ ]:
features = ['Year', 'Lag_1', 'Lag_2', 'Lag_3', 'Rolling_3yr_avg', 'Rolling_5yr_avg']
target   = 'SPPOPGROWUSA'

X = df[features]
y = df[target]

# shuffle=False keeps time order intact (important for time series!)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=False
)

print(f'Training samples : {len(X_train)}')
print(f'Testing  samples : {len(X_test)}')

## Step 6 — Train 3 ML Models

> We try 3 different algorithms and compare which works best:
> - **Linear Regression** — draws the best straight line through data
> - **Random Forest** — combines many decision trees for better accuracy
> - **Gradient Boosting** — builds trees one at a time, each fixing errors of the last

In [ ]:
models = {
    'Linear Regression'  : LinearRegression(),
    'Random Forest'      : RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting'  : GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)

    results[name] = {'model': model, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'preds': y_pred}

    print(f'🤖 {name}')
    print(f'   MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}')
    print()

## Step 7 — Compare Models Visually

In [ ]:
best_name  = min(results, key=lambda k: results[k]['RMSE'])
best_model = results[best_name]['model']
print(f'🏆 Best model: {best_name}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
ax = axes[0]
ax.plot(y_test.values, label='Actual',    color='steelblue',  linewidth=2)
ax.plot(results[best_name]['preds'], label='Predicted', color='orangered', linewidth=2, linestyle='--')
ax.set_title(f'Actual vs Predicted — {best_name}', fontweight='bold')
ax.set_ylabel('Growth Rate (%)')
ax.legend()
ax.grid(True, alpha=0.3)

# R² Bar chart
ax2 = axes[1]
names  = list(results.keys())
r2vals = [results[k]['R2'] for k in names]
colors = ['#4CAF50' if k == best_name else '#90CAF9' for k in names]
bars = ax2.bar(names, r2vals, color=colors, edgecolor='white')
for bar, val in zip(bars, r2vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', fontweight='bold')
ax2.set_title('R² Score Comparison', fontweight='bold')
ax2.set_ylabel('R² (higher = better)')
ax2.set_ylim(0, 1.1)
ax2.tick_params(axis='x', rotation=10)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Step 8 — Forecast 2023–2032 🔮

In [ ]:
future_years = list(range(2023, 2033))
last_known   = df['SPPOPGROWUSA'].values
forecasts    = []
lag1, lag2, lag3 = last_known[-1], last_known[-2], last_known[-3]
roll3 = np.mean(last_known[-3:])
roll5 = np.mean(last_known[-5:])

for yr in future_years:
    inp  = np.array([[yr, lag1, lag2, lag3, roll3, roll5]])
    pred = best_model.predict(inp)[0]
    forecasts.append(pred)
    lag3, lag2, lag1 = lag2, lag1, pred
    roll3 = np.mean(forecasts[-3:]) if len(forecasts) >= 3 else np.mean(forecasts)
    roll5 = np.mean(forecasts[-5:]) if len(forecasts) >= 5 else np.mean(forecasts)

# Print forecast table
print('📅 USA Population Growth Rate Forecast')
print('─' * 35)
for yr, fc in zip(future_years, forecasts):
    bar = '█' * int(fc * 20)
    print(f'  {yr}  {fc:+.4f}%  {bar}')

# Plot
plt.figure(figsize=(13, 5))
plt.plot(df['Year'].values, last_known, color='steelblue', linewidth=2.5, label='Historical (1961–2022)')
plt.plot(future_years, forecasts, color='orangered', linewidth=2.5, linestyle='--',
         marker='o', markersize=6, label='Forecast (2023–2032)')
plt.axvline(x=2022, color='gray', linestyle=':', linewidth=1.5)
plt.title('USA Population Growth Rate — Historical & Forecast', fontsize=13, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Growth Rate (%)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 9 — Save the Best Model

In [ ]:
import os, pickle
os.makedirs('models', exist_ok=True)

with open('models/best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

print(f'✅ Model saved: models/best_model.pkl')
print(f'   Algorithm : {best_name}')
print(f'   R² Score  : {results[best_name]["R2"]:.4f}')

---
## 🎉 Summary

| Step | What we did |
|------|-------------|
| 1 | Loaded the Population-Growth dataset (62 rows, 1961–2022) |
| 2 | Engineered lag & rolling average features |
| 3 | Split data 80% train / 20% test |
| 4 | Trained Linear Regression, Random Forest, Gradient Boosting |
| 5 | Compared models using MAE, RMSE, R² |
| 6 | Forecasted US population growth for 2023–2032 |
| 7 | Saved the best model as `.pkl` |
